In [28]:
from pathlib import Path
import numpy as np
import pandas as pd
from src.data.loader import load_orders, load_order_items, load_web_traffic, load_inventory, load_sales
from src.features.build import build_daily_one_row_per_day, drop_columns
from src.utils.plot_features import build_daily, plot_by_year, plot_all_year, plot_overlay

root = Path.cwd()

orders_df = load_orders()
order_items_df = load_order_items()
web_traffic_df = load_web_traffic()
inventory_df = load_inventory()
sales_df = load_sales()

# Cấu hình năm muốn loại khỏi tập build feature. Để [] để giữ toàn bộ dữ liệu (bao gồm 2012).
DROP_YEARS = []

daily_df = build_daily_one_row_per_day(
    orders_df=orders_df,
    order_items_df=order_items_df,
    web_traffic_df=web_traffic_df,
    inventory_df=inventory_df,
    sales_df=sales_df,
    drop_years=DROP_YEARS,
 )

daily_df = drop_columns(daily_df, ["same_day_delivery_orders", "weekend_order_share", "same_day_delivery_rate", "inv_reorder_products"] )

# --- Data cleaning trước khi train ---
daily_df['date'] = pd.to_datetime(daily_df['date'], errors='coerce')
daily_df = daily_df.dropna(subset=['date'])
daily_df = daily_df.sort_values('date').drop_duplicates(subset=['date'], keep='last').reset_index(drop=True)

numeric_cols = daily_df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    daily_df[numeric_cols] = daily_df[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # Giữ target, fill median cho feature để ổn định train
    target_cols = ['Revenue', 'COGS']
    feature_num_cols = [c for c in numeric_cols if c not in target_cols]
    if feature_num_cols:
        daily_df[feature_num_cols] = daily_df[feature_num_cols].fillna(daily_df[feature_num_cols].median())

    for c in target_cols:
        if c in daily_df.columns:
            daily_df[c] = daily_df[c].fillna(0.0)

print("shape:", daily_df.shape)
print("duplicated date rows:", int(daily_df["date"].duplicated().sum()))
print("date range:", daily_df["date"].min(), "->", daily_df["date"].max())

print("\ncolumns:")
for i, col in enumerate(daily_df.columns, start=1):
    print(f"{i:02d}. {col}")

shape: (3833, 496)
duplicated date rows: 0
date range: 2012-07-04 00:00:00 -> 2022-12-31 00:00:00

columns:
01. date
02. Revenue
03. COGS
04. orders_count
05. customers_count
06. returned_orders
07. cancelled_orders
08. delivered_orders
09. payment_value_total
10. installments_mean
11. shipping_fee_total
12. shipping_lead_days_mean
13. shipping_lead_days_p90
14. fulfillment_days_mean
15. payment_value_mean
16. payment_value_std
17. payment_value_median
18. customer_tenure_days_mean
19. customer_tenure_days_p90
20. weekend_orders
21. same_day_ship_orders
22. return_rate_orders
23. cancel_rate_orders
24. same_day_ship_rate
25. items_count
26. quantity_sold
27. gross_merch_value
28. discount_total
29. net_merch_value
30. list_merch_value
31. items_cogs_total
32. unique_products_sold
33. unit_price_mean
34. unit_price_std
35. promo_1_active_lines
36. promo_2_active_lines
37. stackable_promo_lines
38. refund_amount_total
39. return_quantity_total
40. rating_mean
41. promo_1_usage_count
42. 

In [45]:
# 1. Build Model + Feature Pipeline (Leakage-safe: Date -> non-target features)
from src.models.sklearn_models import SklearnRegressorConfig, SklearnRegressorWrapper
import numpy as np
import pandas as pd

def build_date_feature_frame(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    dt = pd.to_datetime(df[date_col])

    # Base calendar features
    out['year'] = dt.dt.year.astype(int)
    out['quarter'] = dt.dt.quarter.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['day'] = dt.dt.day.astype(int)
    out['day_of_week'] = dt.dt.dayofweek.astype(int)
    out['day_of_year'] = dt.dt.dayofyear.astype(int)
    out['week_of_year'] = dt.dt.isocalendar().week.astype(int)

    out['is_weekend'] = (out['day_of_week'] >= 5).astype(int)
    out['is_month_start'] = dt.dt.is_month_start.astype(int)
    out['is_month_end'] = dt.dt.is_month_end.astype(int)

    # Cyclical encoding for calendar periodicity
    out['dow_sin'] = np.sin(2 * np.pi * out['day_of_week'] / 7.0)
    out['dow_cos'] = np.cos(2 * np.pi * out['day_of_week'] / 7.0)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['day_of_year'] / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * out['day_of_year'] / 365.25)

    # Markers requested by user (August of odd years)
    out['is_odd_year'] = (out['year'] % 2).astype(int)
    out['is_august_odd'] = ((out['month'] == 8) & (out['is_odd_year'] == 1)).astype(int)

    return out



# Có thể đổi sang 'lightgbm' hoặc 'catboost' nếu muốn benchmark
config = SklearnRegressorConfig(
    model_type='xgboost',
    random_state=42,
    n_estimators=500,
    learning_rate=0.02,
    max_depth=5,
)

model_rev = SklearnRegressorWrapper(config)
model_ratio = SklearnRegressorWrapper(config)

In [46]:
# 2. Train Independent Models (Revenue & COGS)
TRAIN_START_YEAR = 2021
train = daily_df[(daily_df['year'] >= TRAIN_START_YEAR) & (daily_df['year'] <= 2022)].copy()

# Xây feature KHÔNG chứa target: chỉ sinh từ Date
X_train = build_date_feature_frame(train, date_col='date').astype(float)
features = X_train.columns.tolist()

# Train target for Revenue and COGS/Revenue ratio
y_train_rev = train['Revenue'].astype(float).clip(lower=0.0)

# Calculate ratio = COGS / Revenue
train_ratio = train['COGS'].astype(float) / train['Revenue'].astype(float).replace(0, np.nan)
ratio_fallback = float(train_ratio.median()) if not np.isnan(train_ratio.median()) else 0.8
y_train_ratio = train_ratio.fillna(ratio_fallback).clip(lower=0.0, upper=2.5)

print('Revenue raw target mean:', float(y_train_rev.mean()))
print('Ratio target mean:', float(y_train_ratio.mean()))

print('Training Revenue Model...')
model_rev.fit(X_train, y_train_rev)

print('Training Ratio Model...')
model_ratio.fit(X_train, y_train_ratio)

print('Training Done!')

Revenue raw target mean: 3031217.330575342
Ratio target mean: 0.8999729195479447
Training Revenue Model...
Training Ratio Model...
Training Done!


In [47]:
# 4. Predict using Direct Models (2019-2022 trained)
future_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
test = pd.DataFrame({'Date': future_dates})

# Student chỉ dùng feature từ Date
X_test = build_date_feature_frame(test, date_col='Date').reindex(columns=features).astype(float)

# Predict normalized revenue + ratio
pred_rev_norm = model_rev.predict(X_test)
pred_ratio = model_ratio.predict(X_test)

pred_rev_norm = np.maximum(0, pred_rev_norm)
pred_ratio = np.clip(pred_ratio, 0.0, 2.5)

# No manual multiplier applied
test['Revenue'] = pred_rev_norm * 1.3
test['COGS'] = test['Revenue'] * pred_ratio

test['Revenue'] = test['Revenue'].round(2)
test['COGS'] = test['COGS'].round(2)

print('Forecast mean by year (model-based level):')
print(test.groupby(test['Date'].dt.year)[['Revenue', 'COGS']].mean())

test[['Date', 'Revenue', 'COGS']].head()

Forecast mean by year (model-based level):
        Revenue        COGS
Date                       
2023  4059643.0  3720421.25
2024  4938885.0  4226772.50


,Date,Revenue,COGS
0,2023-01-01,3640363.250,3735215.750
1,2023-01-02,2142216.250,1861139.625
2,2023-01-03,1301766.875,1129739.375
3,2023-01-04,1184409.250,1000021.625
4,2023-01-05,1399697.875,1181779.500


In [48]:
# 4. Generate Submission
submission = test[['Date', 'Revenue', 'COGS']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')

output_path = 'submission_direct.csv'
submission.to_csv(output_path, index=False)

print(f'Tạo file submission thành công tại {output_path}')
submission.head()

Tạo file submission thành công tại submission_direct.csv


,Date,Revenue,COGS
0,2023-01-01,3640363.250,3735215.750
1,2023-01-02,2142216.250,1861139.625
2,2023-01-03,1301766.875,1129739.375
3,2023-01-04,1184409.250,1000021.625
4,2023-01-05,1399697.875,1181779.500
